# EX_00 — PyTorch vs TensorFlow (ejercicios)

**Notebook de referencia:** `notebook/00_Frameworks_Pytorch_vs_Tensorflow.ipynb`

**Tiempo orientativo:** ~30 minutos.

En esta hoja practicarás ideas equivalentes en ambos frameworks: tensores, capas lineales y un forward pass mínimo.


## Actividad 1 — Activación a mano

Implementa en NumPy una función `relu` y otra `sigmoid` y comprueba que coinciden con `torch` y `tensorflow` en un vector de prueba.

*Hint:* use `torch.relu`, `tf.nn.relu`; for sigmoid use `torch.sigmoid` and `tf.nn.sigmoid`.


In [2]:
import numpy as np
import torch
import tensorflow as tf

# TODO: implement relu_np(z) and sigmoid_np(z)
def relu_np(z):
    return np.maximum(0, z)

def sigmoid_np(z):
    return 1 / (1 + np.exp(-z))

x = np.array([-2.0, 0.0, 1.5], dtype=np.float32)

# Convertimos el array de numpy a tensores para PyTorch y TensorFlow
x_torch = torch.tensor(x)
x_tf = tf.constant(x)

# TODO: assert close to torch and tensorflow outputs
# Calculamos las salidas con nuestras funciones de NumPy
relu_out = relu_np(x)
sigmoid_out = sigmoid_np(x)

# Comprobamos que coinciden con PyTorch
np.testing.assert_allclose(relu_out, torch.relu(x_torch).numpy(), rtol=1e-5, atol=1e-8)
np.testing.assert_allclose(sigmoid_out, torch.sigmoid(x_torch).numpy(), rtol=1e-5, atol=1e-8)

# Comprobamos que coinciden con TensorFlow
np.testing.assert_allclose(relu_out, tf.nn.relu(x_tf).numpy(), rtol=1e-5, atol=1e-8)
np.testing.assert_allclose(sigmoid_out, tf.nn.sigmoid(x_tf).numpy(), rtol=1e-5, atol=1e-8)

print("¡Todas las comprobaciones han pasado con éxito!")

¡Todas las comprobaciones han pasado con éxito!


## Actividad 2 — Misma arquitectura, dos APIs

Define una red `Linear(10, 3)` + `ReLU` + `Linear(3, 1)` en **PyTorch** (`nn.Sequential`) y la misma en **Keras** (`Sequential`).

*Hint:* set seeds (`torch.manual_seed`, `tf.random.set_seed`) and use explicit init if you want to compare weights.


In [ ]:
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# TODO: build torch_model and keras_model with the architecture above
torch.manual_seed(42)
tf.random.set_seed(42)

#modelo dePytorch
torch_model = nn.Sequential(
nn.Linear(10,3),
nn.ReLU(),
nn.Linear(3,1)
)

#modelo de TensorFlow
tf_model = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(3, activation='relu'),
    layers.Dense(1)
])


# TODO: run a forward pass on random input shape (batch=4, features=10)
# Creamos "tensores" (matrices) con datos aleatorios. 4 ejemplos, cada uno con 10 características.
x_torch = torch.randn(4, 10)
x_tf = tf.random.normal(shape=(4, 10))

# Hacemos una pasada hacia adelante (forward pass) con ambos modelos
torch_output = torch_model(x_torch)
tf_output = tf_model(x_tf)

print("Forma de la salida PyTorch:", torch_output.shape)
print("Forma de la salida Keras:", tf_output.shape)

Forma de la salida PyTorch: torch.Size([4, 1])
Forma de la salida Keras: (4, 1)


## Actividad 3 — Entrenamiento en mini-batch (conceptual + código corto)

Escribe un bucle de **una época** que: (1) muestree un batch sintético `X, y` para regresión, (2) calcule `MSE`, (3) haga `backward` / `gradient` y un paso de optimizador.

Elige **solo uno** de los dos frameworks para el bucle completo; en el otro, documenta en un comentario qué API usarías (`loss.backward`, `tape.gradient`, etc.).


In [10]:
# TODO: one epoch, one framework; comment the other API
import torch
import torch.nn as nn
import torch.optim as optim

# Configuración previa (Modelo dummy y Optimizador)
torch_model = nn.Linear(10, 1) 
optimizer = optim.SGD(torch_model.parameters(), lr=0.01)
criterion = nn.MSELoss()

num_batches = 3
batch_size = 4

print("--- Bucle de 1 Época en PyTorch ---")
for batch_idx in range(num_batches):
    # 1. Muestreo de batch sintético X, y para regresión
    X = torch.randn(batch_size, 10) 
    y_true = torch.randn(batch_size, 1) 

    # 2. Forward pass y cálculo del error (MSE)
    y_pred = torch_model(X)
    loss = criterion(y_pred, y_true)

    # 3. Backward pass y actualización de pesos
    optimizer.zero_grad()  # a) Limpiamos gradientes del batch anterior
    loss.backward()        # b) Calculamos gradientes (derivadas)
    optimizer.step()       # c) Actualizamos los pesos del modelo

    print(f"Batch {batch_idx+1} | MSE Loss: {loss.item():.4f}")


# =====================================================================
# API EQUIVALENTE EN TENSORFLOW / KERAS (Documentación)
# =====================================================================
# En TensorFlow, el cálculo del gradiente no es automático a nivel global,
# requiere grabar las operaciones en una "cinta" (GradientTape).
# 
# El bloque 2 y 3 sería así:
#
# with tf.GradientTape() as tape:
#     y_pred = keras_model(X)
#     loss = tf.keras.losses.MSE(y_true, y_pred)
#
# # Calculamos gradientes a partir de la cinta
# gradients = tape.gradient(loss, keras_model.trainable_variables)
# 
# # Aplicamos los gradientes con el optimizador
# optimizer.apply_gradients(zip(gradients, keras_model.trainable_variables))

--- Bucle de 1 Época en PyTorch ---
Batch 1 | MSE Loss: 0.3302
Batch 2 | MSE Loss: 0.6067
Batch 3 | MSE Loss: 1.3494
